In [ ]:
import json
from pathlib import Path

import requests
from lxml import etree
from SPARQLWrapper import SPARQLWrapper, JSON


# ===================================================================
# Configuration
# ===================================================================

TARGET_ACTS = {
    "GDPR": "32016R0679",
    "NIS2": "32022L2555",
    "DORA": "32022R2554",
    "AI_Act": "32024R1689",
}

DATASET_DIR = Path("EU_DigitalLaw")

SPARQL_ENDPOINT = (
    "https://publications.europa.eu/webapi/rdf/sparql"
)

HEADERS = {
    "User-Agent": "EU-Legal-Research/1.0",
    "Accept": "application/xml,text/xml,*/*",
}


# ===================================================================
# CELLAR / EUR-Lex
# ===================================================================

def get_cellar_url(celex: str) -> str:
    """
    Resolve a CELEX number to its CELLAR resource URL.
    """

    url = (
        "https://publications.europa.eu/resource/celex/"
        + celex
    )

    response = requests.get(
        url,
        headers=HEADERS,
        timeout=60,
        allow_redirects=True,
    )

    response.raise_for_status()

    final_url = response.url

    print(f"CELEX: {celex}")
    print(f"CELLAR URL: {final_url}")

    return final_url


# ===================================================================
# Download legal text
# ===================================================================

def download_legal_text(
    name: str,
    celex: str,
    output_dir: Path,
) -> Path:
    """
    Download the machine-readable XML legal text
    and save it as text.xml.
    """

    resource_url = get_cellar_url(celex)

    response = requests.get(
        resource_url,
        headers={
            "User-Agent": "EU-Legal-Research/1.0",
            "Accept": "application/xml",
        },
        timeout=120,
    )

    response.raise_for_status()

    content = response.content

    # ---------------------------------------------------------------
    # Safety checks
    # ---------------------------------------------------------------

    if not content:
        raise RuntimeError(
            f"EUR-Lex returned an empty response for {celex}"
        )

    print(
        f"Downloaded legal text: "
        f"{len(content):,} bytes"
    )

    # Make sure the response is actually XML.
    try:
        root = etree.fromstring(content)
    except etree.XMLSyntaxError as exc:
        print("Response is not valid XML.")
        print(content[:500])

        raise RuntimeError(
            f"Invalid XML returned for {celex}"
        ) from exc

    print(f"XML root element: {root.tag}")

    # ---------------------------------------------------------------
    # Save
    # ---------------------------------------------------------------

    output_file = output_dir / "text.xml"

    output_file.write_bytes(content)

    print(f"Legal text -> {output_file}")

    return output_file


# ===================================================================
# SPARQL
# ===================================================================

BASIC_METADATA_QUERY = """
PREFIX cdm: <http://publications.europa.eu/ontology/cdm#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT DISTINCT
    ?work
    ?expr
    ?title
    ?dateDoc
    ?dateForce
    ?inForce

WHERE {

    ?work owl:sameAs
        <http://publications.europa.eu/resource/celex/%s> .

    OPTIONAL {
        ?work cdm:work_date_document ?dateDoc .
    }

    OPTIONAL {
        ?work cdm:resource_legal_date_entry-into-force ?dateForce .
    }

    OPTIONAL {
        ?work cdm:resource_legal_in-force ?inForce .
    }

    OPTIONAL {
        ?expr cdm:expression_belongs_to_work ?work ;
              cdm:expression_uses_language
                  <http://publications.europa.eu/resource/authority/language/ENG> ;
              cdm:expression_title ?title .
    }
}
"""


def create_sparql_client() -> SPARQLWrapper:
    """
    Create and configure the SPARQL client.
    """

    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    return sparql


def fetch_metadata(celex: str) -> list[dict]:
    """
    Fetch metadata for a CELEX number.
    """

    query = BASIC_METADATA_QUERY % celex

    sparql = create_sparql_client()
    sparql.setQuery(query)

    results = sparql.query().convert()

    return results["results"]["bindings"]


# ===================================================================
# Clean SPARQL response
# ===================================================================

def clean_row(row: dict) -> dict:
    """
    Convert SPARQL bindings into ordinary Python values.
    """

    return {
        key: value["value"]
        for key, value in row.items()
    }


# ===================================================================
# Save metadata
# ===================================================================

def save_metadata(
    name: str,
    celex: str,
    output_dir: Path,
) -> Path:
    """
    Fetch metadata and save it as metadata.json.
    """

    rows = fetch_metadata(celex)

    metadata = {
        "name": name,
        "celex": celex,
        "source": (
            "EUR-Lex / Publications Office "
            "of the European Union"
        ),
        "sparql_endpoint": SPARQL_ENDPOINT,
        "results": [
            clean_row(row)
            for row in rows
        ],
    }

    output_file = output_dir / "metadata.json"

    with output_file.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            metadata,
            f,
            indent=2,
            ensure_ascii=False,
        )

    print(f"Metadata    -> {output_file}")

    return output_file


# ===================================================================
# Build dataset
# ===================================================================

def build_dataset() -> None:
    """
    Download the legal texts and metadata for all target acts
    and save them in the EU_DigitalLaw dataset structure.
    """

    DATASET_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    for name, celex in TARGET_ACTS.items():

        print()
        print("=" * 80)
        print(f"Processing {name} ({celex})")
        print("=" * 80)

        # -----------------------------------------------------------
        # Create directory
        # -----------------------------------------------------------

        act_dir = DATASET_DIR / name

        act_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        try:

            # -------------------------------------------------------
            # Download legal text
            # -------------------------------------------------------

            download_legal_text(
                name=name,
                celex=celex,
                output_dir=act_dir,
            )

            # -------------------------------------------------------
            # Download metadata
            # -------------------------------------------------------

            save_metadata(
                name=name,
                celex=celex,
                output_dir=act_dir,
            )

            print(f"✓ {name} completed")

        except Exception as exc:

            print()
            print(
                f"ERROR: Could not process "
                f"{name} ({celex})"
            )
            print(exc)


# ===================================================================
# Run
# ===================================================================

if __name__ == "__main__":
    build_dataset()

